# Optional Vision 01 - Convolutional Neural Networks in PyTorch

**Guiding question:** how can one small filter discover a visual pattern wherever it appears, and how do deeper CNNs keep gradients alive?

> **What you already know:** linear layers, autograd, and explicit PyTorch training loops.
> **The gap this chapter closes:** spatial locality, shared filters, receptive fields, and residual gradient routes.
> **What you'll have by the end:** a trained PyTorch CNN on locally generated shapes plus measured filter, feature-map, receptive-field, and skip-path evidence.
> **What this chapter is not:** required preparation for language models.


In [ ]:
# -- Setup and local shape generator ---------------------------------------------
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)
plt.rcParams.update({"figure.facecolor": "#1a1a2e", "axes.facecolor": "#1a1a2e"})

def make_shape(label, size=16, jitter=True):
    image = torch.zeros(size, size)
    shift_x = int(torch.randint(-2, 3, ()).item()) if jitter else 0
    shift_y = int(torch.randint(-2, 3, ()).item()) if jitter else 0
    center = size // 2
    if label == 0:
        column = max(2, min(size - 3, center + shift_x))
        image[2:size - 2, column - 1:column + 1] = 1.0
    else:
        row = max(2, min(size - 3, center + shift_y))
        image[row - 1:row + 1, 2:size - 2] = 1.0
    image += 0.08 * torch.randn_like(image)
    return image.clamp(0, 1)

train_images = torch.stack([make_shape(index % 2) for index in range(320)]).unsqueeze(1)
train_labels = torch.tensor([index % 2 for index in range(320)])
valid_images = torch.stack([make_shape(index % 2) for index in range(80)]).unsqueeze(1)
valid_labels = torch.tensor([index % 2 for index in range(80)])
print(f"Training tensor: {tuple(train_images.shape)}")


## 1 - A sliding dot product reuses one detector

```mermaid
flowchart LR
    A["image patch"] --> B["elementwise multiply"]
    K["shared kernel"] --> B
    B --> C["sum"]
    C --> D["one feature-map pixel"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style K fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```


In [ ]:
# -- Manual convolution pixel and PyTorch agreement ------------------------------
image = valid_images[0:1]
sobel_vertical = torch.tensor(
    [[[-1.0, 0.0, 1.0], [-2.0, 0.0, 2.0], [-1.0, 0.0, 1.0]]]
).unsqueeze(0)
patch = image[0, :, 4:7, 4:7]
manual_pixel = (patch * sobel_vertical[0]).sum()
feature_map = nn.functional.conv2d(image, sobel_vertical)
framework_pixel = feature_map[0, 0, 4, 4]
print(f"Manual sliding dot product: {manual_pixel.item():.6f}")
print(f"PyTorch feature-map pixel:  {framework_pixel.item():.6f}")
print(f"Agreement: {torch.allclose(manual_pixel, framework_pixel)}")
assert torch.allclose(manual_pixel, framework_pixel)


In [ ]:
# -- Sobel evidence: detector location changes, weights do not -------------------
horizontal_image = make_shape(1, jitter=False).unsqueeze(0).unsqueeze(0)
vertical_image = make_shape(0, jitter=False).unsqueeze(0).unsqueeze(0)
vertical_response = nn.functional.conv2d(vertical_image, sobel_vertical).abs()
horizontal_response = nn.functional.conv2d(horizontal_image, sobel_vertical).abs()

fig, axes = plt.subplots(2, 2, figsize=(8, 7))
axes[0, 0].imshow(vertical_image[0, 0], cmap="gray")
axes[0, 1].imshow(vertical_response[0, 0], cmap="magma")
axes[1, 0].imshow(horizontal_image[0, 0], cmap="gray")
axes[1, 1].imshow(horizontal_response[0, 0], cmap="magma")
for axis in axes.flat:
    axis.axis("off")
plt.show()
print(f"Vertical-shape response sum:   {vertical_response.sum().item():.3f}")
print(f"Horizontal-shape response sum: {horizontal_response.sum().item():.3f}")


## 2 - Stride and pooling trade detail for reach

**Predict:** which operation reduces width from 16 to 8 without learning new weights: stride-two convolution or max-pooling? Both can; their evidence is different.


In [ ]:
# -- Stride and pooling shape evidence -------------------------------------------
random_filter = torch.randn(4, 1, 3, 3)
stride_one = nn.functional.conv2d(image, random_filter, padding=1, stride=1)
stride_two = nn.functional.conv2d(image, random_filter, padding=1, stride=2)
pooled = nn.functional.max_pool2d(stride_one, kernel_size=2)
print(f"Stride 1: {tuple(stride_one.shape)}")
print(f"Stride 2: {tuple(stride_two.shape)}")
print(f"Pool after stride 1: {tuple(pooled.shape)}")
print("Takeaway: both shrink space; pooling summarizes an existing map.")


## 3 - Train filters, then inspect their specialization

```mermaid
flowchart LR
    A["local shape images"] --> B["learned conv filters"]
    B --> C["feature maps"]
    C --> D["global average"]
    D --> E["vertical or horizontal"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```


In [ ]:
# -- Small CNN and explicit training loop ----------------------------------------
class ShapeCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 6, 3, padding=1)
        self.conv2 = nn.Conv2d(6, 10, 3, padding=1)
        self.head = nn.Linear(10, 2)

    def features(self, images):
        first = torch.relu(self.conv1(images))
        second = torch.relu(self.conv2(nn.functional.max_pool2d(first, 2)))
        return first, second

    def forward(self, images):
        _, second = self.features(images)
        pooled = second.mean(dim=(-2, -1))
        return self.head(pooled)

model = ShapeCNN()
loader = DataLoader(TensorDataset(train_images, train_labels), batch_size=32, shuffle=True)
optimizer = torch.optim.Adam(model.parameters(), lr=0.02)
losses = []
for epoch in range(30):
    for inputs, labels in loader:
        logits = model(inputs)
        loss = nn.functional.cross_entropy(logits, labels)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
    losses.append(float(loss.detach()))

with torch.inference_mode():
    accuracy = (model(valid_images).argmax(dim=-1) == valid_labels).float().mean().item()
print(f"Validation accuracy: {accuracy:.3f}")
print(f"Loss: {losses[0]:.4f} -> {losses[-1]:.4f}")
assert accuracy > 0.90


In [ ]:
# -- Learned filters and feature maps --------------------------------------------
filters = model.conv1.weight.detach()
first_maps, second_maps = model.features(valid_images[:1])
fig, axes = plt.subplots(2, 6, figsize=(13, 5))
for index in range(6):
    axes[0, index].imshow(filters[index, 0], cmap="coolwarm")
    axes[0, index].set_title(f"filter {index}")
    axes[1, index].imshow(first_maps[0, index].detach(), cmap="magma")
    axes[1, index].set_title(f"map {index}")
    axes[0, index].axis("off")
    axes[1, index].axis("off")
plt.show()
print("Takeaway: training specializes shared filters; maps show where each responds.")


## 4 - Receptive field grows through stacked local steps

A pixel in the second convolution depends on more input pixels than a pixel in the first. Autograd can measure the exact route.


In [ ]:
# -- Receptive field via input gradient -------------------------------------------
probe = valid_images[:1].clone().requires_grad_(True)
first = torch.relu(model.conv1(probe))
pooled = nn.functional.max_pool2d(first, 2)
second_pre_activation = model.conv2(pooled)
center_row = second_pre_activation.size(-2) // 2
center_column = second_pre_activation.size(-1) // 2
center_values = second_pre_activation[0, :, center_row, center_column]
strongest_channel = int(center_values.abs().argmax())
center_values[strongest_channel].backward()
receptive_mask = probe.grad[0, 0].abs() > 1e-9
rows, columns = torch.where(receptive_mask)
height = int(rows.max() - rows.min() + 1)
width = int(columns.max() - columns.min() + 1)
print(f"Traced pre-activation channel: {strongest_channel}")
print(f"Measured receptive field bounding box: {height} x {width}")
assert height > 3 and width > 3


## 5 - Residual shortcuts create another gradient route

```mermaid
flowchart LR
    X["input x"] --> F["two learned transforms"]
    X --> S["identity shortcut"]
    F --> A["add"]
    S --> A
    A --> O["output"]
    style X fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style S fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style A fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style O fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```


In [ ]:
# -- ResNet gradient proof: same transforms, one extra identity route ------------
torch.manual_seed(9)
plain_layers = nn.ModuleList([nn.Linear(12, 12, bias=False) for _ in range(12)])
residual_layers = nn.ModuleList([nn.Linear(12, 12, bias=False) for _ in range(12)])
for plain, residual in zip(plain_layers, residual_layers):
    residual.weight.data.copy_(plain.weight.data)
    plain.weight.data.mul_(0.25)
    residual.weight.data.mul_(0.25)

x_plain = torch.randn(1, 12, requires_grad=True)
x_residual = x_plain.detach().clone().requires_grad_(True)
plain = x_plain
residual = x_residual
for plain_layer, residual_layer in zip(plain_layers, residual_layers):
    plain = torch.tanh(plain_layer(plain))
    residual = residual + torch.tanh(residual_layer(residual))
plain.sum().backward()
residual.sum().backward()

plain_norm = x_plain.grad.norm().item()
residual_norm = x_residual.grad.norm().item()
print(f"Input gradient without shortcut: {plain_norm:.8f}")
print(f"Input gradient with shortcut:    {residual_norm:.8f}")
print(f"Measured route advantage: {residual_norm / max(plain_norm, 1e-12):,.1f}x")
assert residual_norm > plain_norm


## 6 - Coverage ledger

**Built and measured:** convolution, Sobel response, stride, pooling, trained filters, feature maps, receptive field, ResNet gradient route.

**Explained or illustrated:** locality and weight sharing.

**Named and out of scope:** large pretrained backbones, transfer learning, and detection.

Next, [the autoencoder lab](02-pytorch-autoencoders.ipynb) asks whether a compact latent code can preserve enough visual evidence to reconstruct the image.
